# Capa Gold – Modelo dimensional (estrella) · Consumo Alumbrado Público (Sep 2026)

Consume las tablas de `workspace.fenomeno_nino_silver` y publica en `workspace.fenomeno_nino_gold` un **modelo estrella listo para Power BI**:

* `dim_concepto` — activo/concepto medido (puentes, canchas, arquitectónicos…) con potencia, cantidad y horas.
* `dim_escenario` — escenario de reducción horaria (base, −1h, −2h, −3h, −4h).
* `fact_consumo_escenarios` — granularidad concepto × escenario: kWh, valor $, ahorro kWh, ahorro $ y % de ahorro.
* `fact_ajustes_factura` — ajustes de energía aplicados en la factura (descuentos por temporización).

**Relaciones en Power BI**: `dim_concepto.concepto` → `fact_consumo_escenarios.concepto` y `dim_escenario.escenario_id` → `fact_consumo_escenarios.escenario_id` (1 a muchos). `fact_ajustes_factura` funciona como tabla autónoma o relacionada por su propio texto de concepto.

## 1. Dimensiones

* `dim_escenario`: 5 filas (escenario base + reducciones de 1 a 4 h/día).
* `dim_concepto`: una fila por concepto del análisis, con categoría derivada (`puente`, `canchas`, `arquitectónicos`, `subtotal`, `total`).

In [0]:
GOLD = "workspace.fenomeno_nino_gold"
SILVER = "workspace.fenomeno_nino_silver"
PERIODO = "2026-09"

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_escenario AS
SELECT 0 AS escenario_id, 'Base (sin reducción)' AS escenario, 0 AS horas_reduccion
UNION ALL SELECT 1, 'Reducción -1 h/día', 1
UNION ALL SELECT 2, 'Reducción -2 h/día', 2
UNION ALL SELECT 3, 'Reducción -3 h/día', 3
UNION ALL SELECT 4, 'Reducción -4 h/día', 4
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_concepto AS
SELECT
  concepto,
  CASE
    WHEN concepto = 'TOTAL' THEN 'total'
    WHEN concepto LIKE 'Subtotal%' THEN 'subtotal'
    WHEN concepto LIKE '%Canchas%' THEN 'canchas'
    WHEN concepto LIKE '%Arquitect%' THEN 'arquitectonicos'
    WHEN concepto LIKE '%Puente%' THEN 'puente'
    ELSE 'otros'
  END AS categoria,
  potencia_w, cantidad, carga_kw, horas_dia_base,
  es_total, es_subtotal, nota_imputacion
FROM {SILVER}.consumo_escenarios
""")

print("dim_escenario →", spark.table(f"{GOLD}.dim_escenario").count(), "filas")
print("dim_concepto →", spark.table(f"{GOLD}.dim_concepto").count(), "filas")
display(spark.table(f"{GOLD}.dim_concepto").select("concepto", "categoria", "potencia_w", "cantidad", "horas_dia_base"))

## 2. `fact_consumo_escenarios`

Unpivote de las tablas de escenario de Silver: de 5 columnas (base, −1h…−4h) a **una fila por concepto × escenario** (7 × 5 = 35 filas), con medidas de consumo y ahorro. Incluye banderas `es_total`/`es_subtotal` para que Power BI pueda excluirlas de las sumas.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_consumo_escenarios AS
WITH base AS (
  SELECT ce.concepto, 0 AS escenario_id,
         ce.kwh_base AS kwh_mes, va.valor_base AS valor_pesos,
         0.0 AS ahorro_kwh, 0.0 AS ahorro_pesos, 0.0 AS pct_ahorro,
         ce.es_total, ce.es_subtotal
  FROM {SILVER}.consumo_escenarios ce
  JOIN {SILVER}.valor_escenarios va USING (concepto)
),
e1 AS (
  SELECT ce.concepto, 1 AS escenario_id,
         ce.kwh_menos_1h AS kwh_mes, va.valor_menos_1h AS valor_pesos,
         ah.ahorro_kwh_1h AS ahorro_kwh, va.ahorro_pesos_1h AS ahorro_pesos, ah.pct_1h AS pct_ahorro,
         ce.es_total, ce.es_subtotal
  FROM {SILVER}.consumo_escenarios ce
  JOIN {SILVER}.valor_escenarios va USING (concepto)
  JOIN {SILVER}.ahorro_escenarios ah USING (concepto)
),
e2 AS (
  SELECT ce.concepto, 2 AS escenario_id,
         ce.kwh_menos_2h AS kwh_mes, va.valor_menos_2h AS valor_pesos,
         ah.ahorro_kwh_2h AS ahorro_kwh, va.ahorro_pesos_2h AS ahorro_pesos, ah.pct_2h AS pct_ahorro,
         ce.es_total, ce.es_subtotal
  FROM {SILVER}.consumo_escenarios ce
  JOIN {SILVER}.valor_escenarios va USING (concepto)
  JOIN {SILVER}.ahorro_escenarios ah USING (concepto)
),
e3 AS (
  SELECT ce.concepto, 3 AS escenario_id,
         ce.kwh_menos_3h AS kwh_mes, va.valor_menos_3h AS valor_pesos,
         ah.ahorro_kwh_3h AS ahorro_kwh, va.ahorro_pesos_3h AS ahorro_pesos, ah.pct_3h AS pct_ahorro,
         ce.es_total, ce.es_subtotal
  FROM {SILVER}.consumo_escenarios ce
  JOIN {SILVER}.valor_escenarios va USING (concepto)
  JOIN {SILVER}.ahorro_escenarios ah USING (concepto)
),
e4 AS (
  SELECT ce.concepto, 4 AS escenario_id,
         ce.kwh_menos_4h AS kwh_mes, va.valor_menos_4h AS valor_pesos,
         ah.ahorro_kwh_4h AS ahorro_kwh, va.ahorro_pesos_4h AS ahorro_pesos, ah.pct_4h AS pct_ahorro,
         ce.es_total, ce.es_subtotal
  FROM {SILVER}.consumo_escenarios ce
  JOIN {SILVER}.valor_escenarios va USING (concepto)
  JOIN {SILVER}.ahorro_escenarios ah USING (concepto)
)
SELECT *, '{PERIODO}' AS periodo FROM base
UNION ALL SELECT *, '{PERIODO}' FROM e1
UNION ALL SELECT *, '{PERIODO}' FROM e2
UNION ALL SELECT *, '{PERIODO}' FROM e3
UNION ALL SELECT *, '{PERIODO}' FROM e4
""")

print("fact_consumo_escenarios →", spark.table(f"{GOLD}.fact_consumo_escenarios").count(), "filas")
display(spark.table(f"{GOLD}.fact_consumo_escenarios"))

## 3. `fact_ajustes_factura`

Ajustes de energía aplicados en la factura de septiembre (negativos = descuentos), con su tipo de registro para filtrar la fila TOTAL en Power BI.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_ajustes_factura AS
SELECT
  concepto,
  CASE
    WHEN concepto = 'TOTAL' THEN 'total'
    WHEN concepto LIKE '%Canchas%' THEN 'canchas'
    WHEN concepto LIKE '%Arquitect%' THEN 'arquitectonicos'
    WHEN concepto LIKE '%Puente%' THEN 'puente'
    ELSE 'otros'
  END AS categoria,
  hoja_origen, puntos_luminosos, kwh_mes_factura, kwh_stn2, valor_pesos,
  tipo_registro, observacion, '{PERIODO}' AS periodo
FROM {SILVER}.ajustes_factura
""")

print("fact_ajustes_factura →", spark.table(f"{GOLD}.fact_ajustes_factura").count(), "filas")
display(spark.table(f"{GOLD}.fact_ajustes_factura"))

## 4. Validación del modelo

Controles contra los totales del archivo: número de filas esperadas por tabla y sumas de medidas (excluyendo filas TOTAL/Subtotal para no duplicar).

In [0]:
import pandas as pd

resultados = []
def check(nombre, real, esperado, tol=1e-3):
    ok = abs(real - esperado) <= tol * max(1.0, abs(esperado))
    resultados.append({"control": nombre, "real": real, "esperado": esperado, "estado": "OK" if ok else "FALLA"})

def escalar(sql):
    return spark.sql(sql).first()[0]

# Conteos
check("filas dim_escenario", escalar(f"SELECT COUNT(*) FROM {GOLD}.dim_escenario"), 5, tol=0)
check("filas dim_concepto", escalar(f"SELECT COUNT(*) FROM {GOLD}.dim_concepto"), 7, tol=0)
check("filas fact_consumo (7×5)", escalar(f"SELECT COUNT(*) FROM {GOLD}.fact_consumo_escenarios"), 35, tol=0)
check("filas fact_ajustes", escalar(f"SELECT COUNT(*) FROM {GOLD}.fact_ajustes_factura"), 5, tol=0)

# Medidas (sin dobles conteo: excluyen filas TOTAL/Subtotal)
check("Σ valor base (conceptos)", escalar(f"""
  SELECT SUM(valor_pesos) FROM {GOLD}.fact_consumo_escenarios
  WHERE escenario_id = 0 AND NOT es_total AND NOT es_subtotal"""), 4606783.324032)
check("Σ ahorro $ -4h (conceptos)", escalar(f"""
  SELECT SUM(ahorro_pesos) FROM {GOLD}.fact_consumo_escenarios
  WHERE escenario_id = 4 AND NOT es_total AND NOT es_subtotal"""), 3499991.338319)
check("Σ kWh base (conceptos)", escalar(f"""
  SELECT SUM(kwh_mes) FROM {GOLD}.fact_consumo_escenarios
  WHERE escenario_id = 0 AND NOT es_total AND NOT es_subtotal"""), 6886.5)
check("Σ kWh ajustes (sin TOTAL)", escalar(f"""
  SELECT SUM(kwh_mes_factura) FROM {GOLD}.fact_ajustes_factura
  WHERE tipo_registro = 'ajuste'"""), -194188.402294)
check("Σ puntos luminosos ajustes", escalar(f"""
  SELECT SUM(puntos_luminosos) FROM {GOLD}.fact_ajustes_factura
  WHERE tipo_registro = 'ajuste'"""), 4209, tol=0)

# Coherencia interna: fila TOTAL del fact = suma de conceptos
check("fila TOTAL = Σ conceptos (valor base)", escalar(f"""
  SELECT MAX(valor_pesos) FROM {GOLD}.fact_consumo_escenarios
  WHERE escenario_id = 0 AND es_total""") - escalar(f"""
  SELECT SUM(valor_pesos) FROM {GOLD}.fact_consumo_escenarios
  WHERE escenario_id = 0 AND NOT es_total AND NOT es_subtotal"""), 0.0)

df_check = pd.DataFrame(resultados)
display(df_check)
assert (df_check["estado"] == "OK").all(), f"Controles fallidos:\\n{df_check[df_check.estado != 'OK']}"
print(f"\\n{len(resultados)} controles OK — capa Gold validada.")

## 5. Conexión desde Power BI

1. En Power BI Desktop: **Obtener datos → Databricks** (conector de Databricks), con el hostname del workspace y autenticación Azure AD.
2. Importar (o DirectQuery) las tablas de `workspace.fenomeno_nino_gold`: `dim_concepto`, `dim_escenario`, `fact_consumo_escenarios`, `fact_ajustes_factura`.
3. Relaciones sugeridas (1 → *):
   * `dim_concepto[concepto]` → `fact_consumo_escenarios[concepto]`
   * `dim_escenario[escenario_id]` → `fact_consumo_escenarios[escenario_id]`
4. En los visuales, filtrar `es_total = False` y `es_subtotal = False` para evitar dobles conteo; la fila TOTAL queda disponible para control.